In [3]:
spark.stop()

In [1]:
import sys
sys.path.append('/home/azureuser/prathyusha/Kearney/prathyusha')
from utils import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window
spark = instantiate_spark_sedona("10g")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/10 12:34:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/10 12:34:41 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).
26/02/10 12:34:42 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/10 12:34:42 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/02/10 12:34:42 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Spark Session and SedonaContext have been successfully initiated.


In [10]:
spark.conf.set("spark.sql.session.timeZone", "Asia/Kolkata")

loc = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet("abfss://propheus-data-staging@propheusdatabay.dfs.core.windows.net/kearney/propheus-rtb-data/sample-data/*")
)

loc = loc.withColumn("timestamp_ist", to_timestamp(col("timestamp") / 1000)).drop('consent_string')

loc.show(5)


+-------------+--------------------+------------+----+------------------+-----------------+------------+-------+-------+-----------+----------+------------+------+-----------+-------+-----------------+------+------------------+-------+----------+--------+----------------+--------------------+--------------------+-----+--------------------+-----------------+----------+--------------------+
|    timestamp|                maid|        ipv4|ipv6|          latitude|        longitude|country_code| region|   city|    carrier|connection|network_type|mccmnc|        isp|   make|            model|device|             brand|     os|os_version|language|          bundle|            app_name|                  ua| gdpr|carrier_country_code|     carrier_name|javascript|       timestamp_ist|
+-------------+--------------------+------------+----+------------------+-----------------+------------+-------+-------+-----------+----------+------------+------+-----------+-------+-----------------+------+--------

In [11]:
loc.printSchema()

root
 |-- timestamp: long (nullable = true)
 |-- maid: string (nullable = true)
 |-- ipv4: string (nullable = true)
 |-- ipv6: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- country_code: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- carrier: string (nullable = true)
 |-- connection: string (nullable = true)
 |-- network_type: string (nullable = true)
 |-- mccmnc: string (nullable = true)
 |-- isp: string (nullable = true)
 |-- make: string (nullable = true)
 |-- model: string (nullable = true)
 |-- device: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- os: string (nullable = true)
 |-- os_version: string (nullable = true)
 |-- language: string (nullable = true)
 |-- bundle: string (nullable = true)
 |-- app_name: string (nullable = true)
 |-- ua: string (nullable = true)
 |-- gdpr: boolean (nullable = true)
 |-- carrier_country_code: string (nul

In [12]:
loc.select(col('isp')).distinct().show()

[Stage 18:====================================================>   (45 + 3) / 48]

+--------------------+
|                 isp|
+--------------------+
|         True Mobile|
|           MTN Sudan|
|Rajamangala Unive...|
| Shaw Communications|
|Mahasarakham Univ...|
|Proen Corp Public...|
|Triple T Broadban...|
|               TELUS|
|     Microsoft Azure|
|Ubonratchathani R...|
|            Symphony|
|Die Schweizerisch...|
|Thai Customs Depa...|
|Tobacco Authority...|
|                 TOT|
| Ministry of Culture|
|           AIS Fibre|
|                  01|
|Mass Rapid Transi...|
|CLOUDFOREST CO., ...|
+--------------------+
only showing top 20 rows



In [13]:
loc.select(col('isp')).distinct().count()

643

In [15]:
loc.filter(
    col("isp").isNotNull() &
    col("isp").isin("AIS Fibre", "3BB Broadband")
).distinct().show()


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/home/azureuser/prathyusha/Kearney/vishaal/data_eda/.venv/lib/python3.10/site-packages/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/azureuser/prathyusha/Kearney/vishaal/data_eda/.venv/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/azureuser/prathyusha/Kearney/vishaal/data_eda/.venv/lib/python3.10/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


Py4JError: functions does not exist in the JVM